# Date-range and multi-page face-entry cleanup

This notebook loads the deduplicated Economist face-detection CSV, removes rows outside the 1940-2007 analysis period, counts how many rows are tied to source scans spanning 1, 2, 3, etc. pages, and writes a cleaned CSV that removes remaining entries spanning more than two pages. It also writes those removed in-period entries to a separate CSV so the full 1940-2007 set can be reconstructed.

The page span is read from the filename stem. For example, `1996-0413-0045,0046_...jpg` is treated as a two-page source scan because the encoded page list contains `0045,0046`.

## Input and output paths

Paths are relative to `code/scripts`, which is the intended working directory for this notebook.

In [ ]:
from pathlib import Path

import pandas as pd


deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
cleaned_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned.csv")
dropped_over_two_pages_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_dropped-over-two-pages.csv")

assert deduplicated_csv.exists(), f"Missing input CSV: {deduplicated_csv}"
cleaned_csv.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {deduplicated_csv}")
print(f"Cleaned output: {cleaned_csv}")
print(f"Dropped-entry output: {dropped_over_two_pages_csv}")

## Load and validate data

The cleaned output keeps the same columns as the deduplicated input. Temporary parsing fields are used only inside this notebook.

In [ ]:
faces = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})

expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated face CSV is empty."
assert faces["Filename"].notna().all(), "Filename must be present for every row."

print(f"Loaded {len(faces):,} deduplicated face entries.")
faces.head()

## Parse year and involved page count

The filename begins with `yyyy-mmdd-pppp` for single-page scans and `yyyy-mmdd-pppp,pppp,...` for multi-page scans. The initial four digits provide the issue year, and counting the comma-separated page tokens gives the number of involved pages for each row.

In [ ]:
filename_parts = faces["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)

unparsed_filenames = faces.loc[filename_parts["source_pages"].isna(), "Filename"]
assert unparsed_filenames.empty, unparsed_filenames.head().tolist()

faces_with_fields = faces.assign(
    issue_year=filename_parts["issue_id"].str.slice(0, 4).astype("int64"),
    source_pages=filename_parts["source_pages"],
    involved_page_count=filename_parts["source_pages"].str.split(",").str.len().astype("int64"),
)

assert faces_with_fields["issue_year"].between(1800, 2100).all()
assert faces_with_fields["involved_page_count"].ge(1).all()
faces_with_fields[["Filename", "issue_year", "source_pages", "involved_page_count"]].head()

## Remove entries outside 1940-2007

The thesis analysis period runs from 1940 through 2007, inclusive. Rows before 1940 and after 2007 are counted separately before they are excluded.

In [ ]:
pre_1940_mask = faces_with_fields["issue_year"] < 1940
post_2007_mask = faces_with_fields["issue_year"] > 2007
analysis_period_mask = faces_with_fields["issue_year"].between(1940, 2007, inclusive="both")

pre_1940_entries = int(pre_1940_mask.sum())
post_2007_entries = int(post_2007_mask.sum())
analysis_period_entries = int(analysis_period_mask.sum())

assert pre_1940_entries + post_2007_entries + analysis_period_entries == len(faces_with_fields)

year_filter_summary = pd.DataFrame(
    [
        {"category": "before_1940", "entry_count": pre_1940_entries},
        {"category": "after_2007", "entry_count": post_2007_entries},
        {"category": "kept_1940_2007", "entry_count": analysis_period_entries},
    ]
)

print("Year filter summary:")
print(year_filter_summary.to_string(index=False))

analysis_period_faces = faces_with_fields.loc[analysis_period_mask].copy()
assert analysis_period_faces["issue_year"].between(1940, 2007, inclusive="both").all()

year_filter_summary

## Textual page-span summary

After the year filter, this table reports the number of remaining deduplicated entries for each page span. Rows with page counts greater than 2 are removed from the final cleaned CSV; two-page entries are retained.

In [ ]:
span_summary = (
    analysis_period_faces.groupby("involved_page_count", as_index=False)
    .size()
    .rename(columns={"size": "entry_count"})
    .sort_values("involved_page_count", kind="mergesort")
)
span_summary["share_of_entries"] = span_summary["entry_count"] / len(analysis_period_faces)
span_summary["is_multi_page"] = span_summary["involved_page_count"] > 1

print("Entries by number of involved pages:")
print(
    span_summary.to_string(
        index=False,
        formatters={"share_of_entries": "{:.4%}".format},
    )
)

multi_page_entries = int(span_summary.loc[span_summary["is_multi_page"], "entry_count"].sum())
entries_over_two_pages_to_drop = int(
    span_summary.loc[span_summary["involved_page_count"] > 2, "entry_count"].sum()
)

print()
print(f"Multi-page entries: {multi_page_entries:,}")
print(f"Entries spanning more than 2 pages to drop: {entries_over_two_pages_to_drop:,}")

span_summary

## Drop entries spanning more than two pages

The cleaned dataset preserves entries from 1940 through 2007 that are tied to one or two encoded source pages. Entries spanning three or more pages are removed.

In [ ]:
keep_page_span_mask = analysis_period_faces["involved_page_count"] <= 2
cleaned_faces = analysis_period_faces.loc[keep_page_span_mask, expected_columns].copy()
dropped_over_two_page_faces = analysis_period_faces.loc[~keep_page_span_mask, expected_columns].copy()

assert len(cleaned_faces) + len(dropped_over_two_page_faces) == len(analysis_period_faces)
assert len(dropped_over_two_page_faces) == entries_over_two_pages_to_drop
assert list(cleaned_faces.columns) == expected_columns

print(f"Kept entries:    {len(cleaned_faces):,}")
print(f"Dropped entries: {len(dropped_over_two_page_faces):,}")

dropped_over_two_page_faces.head()

## Write and verify output CSVs

The cleaned CSV has the same schema as the deduplicated input and contains only 1940-2007 entries tied to one or two encoded source pages. A companion CSV preserves the in-period entries removed because they span three or more encoded source pages. Together, the two files can be concatenated to reconstruct the full 1940-2007 set.

In [ ]:
cleaned_faces.to_csv(cleaned_csv, index=False)
dropped_over_two_page_faces.to_csv(dropped_over_two_pages_csv, index=False)

reloaded_cleaned = pd.read_csv(cleaned_csv, dtype={"Filename": "string"})
reloaded_dropped = pd.read_csv(dropped_over_two_pages_csv, dtype={"Filename": "string"})
reloaded_parts = reloaded_cleaned["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)
reloaded_page_counts = reloaded_parts["source_pages"].str.split(",").str.len().astype("int64")
reloaded_years = reloaded_parts["issue_id"].str.slice(0, 4).astype("int64")

assert list(reloaded_cleaned.columns) == expected_columns
assert list(reloaded_dropped.columns) == expected_columns
assert len(reloaded_cleaned) == len(cleaned_faces)
assert len(reloaded_dropped) == len(dropped_over_two_page_faces)
assert reloaded_years.between(1940, 2007, inclusive="both").all()
assert reloaded_page_counts.le(2).all()

reloaded_dropped_parts = reloaded_dropped["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)
reloaded_dropped_years = reloaded_dropped_parts["issue_id"].str.slice(0, 4).astype("int64")
reloaded_dropped_page_counts = reloaded_dropped_parts["source_pages"].str.split(",").str.len().astype("int64")

assert reloaded_dropped_years.between(1940, 2007, inclusive="both").all()
assert reloaded_dropped_page_counts.gt(2).all()
assert len(reloaded_cleaned) + len(reloaded_dropped) == len(analysis_period_faces)

print(f"Wrote {len(reloaded_cleaned):,} cleaned entries to {cleaned_csv}")
print(f"Wrote {len(reloaded_dropped):,} dropped entries to {dropped_over_two_pages_csv}")